# 01 Data Acquisition

This notebook loads the source manufacturing dataset, builds the job-level model-ready table, calculates net cross-sectional area and shear load, and saves `data/processed_material_database.csv`.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import os

print("Imports loaded successfully.")

from pathlib import Path
import os

# Notebook is inside notebooks/, so ../data goes to the repo data folder
data_dir = Path("../data")

excel_path = data_dir / "Materials_Database.xlsx"
processed_path = data_dir / "processed_material_database.csv"

print("Current working directory:")
print(os.getcwd())

print("\nFiles in ../data:")
print(os.listdir(data_dir))

print("\nLooking for Excel file:")
print(excel_path)
print("Excel file exists:", excel_path.exists())

print("\nLooking for processed CSV:")
print(processed_path)
print("Processed CSV exists:", processed_path.exists())


In [ ]:
if excel_path.exists():
    print("Raw workbook found. Loading Excel sheets...")

    lot = pd.read_excel(excel_path, sheet_name="LotInfo_15-5")
    shear = pd.read_excel(excel_path, sheet_name="ShearResults_15-5")
    heat_treat = pd.read_excel(excel_path, sheet_name="Heat Treat_15-5")
    summary = pd.read_excel(excel_path, sheet_name="JobSummary_15-5")

    print("Sheets loaded:")
    print(f"LotInfo_15-5:      {lot.shape}")
    print(f"ShearResults_15-5: {shear.shape}")
    print(f"Heat Treat_15-5:   {heat_treat.shape}")
    print(f"JobSummary_15-5:   {summary.shape}")

else:
    raise FileNotFoundError(
        "Raw workbook not found. Check that Materials_Database.xlsx is in the data folder."
    )

In [ ]:
processed_df = summary.merge(
    lot,
    on="Lot_ID",
    how="left",
    suffixes=("", "_lot")
)

if "Mean_OD_in" in processed_df.columns and "Mean_ID_in" in processed_df.columns:
    processed_df["MeanNetArea_in2"] = (
        np.pi / 4
    ) * (processed_df["Mean_OD_in"]**2 - processed_df["Mean_ID_in"]**2)


if "MeanShear_ksi" in processed_df.columns and "MeanNetArea_in2" in processed_df.columns:
    processed_df["MeanShearLoad_lbf"] = (
        processed_df["MeanShear_ksi"] * processed_df["MeanNetArea_in2"] * 1000 * 2
    )

processed_df.to_csv(processed_path, index=False)

print("Processed dataset created and saved.")
print(f"Processed dataset shape: {processed_df.shape}")
print(f"Saved to: {processed_path}")

## Preview processed dataset

In [ ]:
print("Processed dataset shape:")
print(processed_df.shape)

display_cols = [
    "JobNum", "Lot_ID", "PartNum", "TestStage", "AgeTemp_F", "AgeTime_hr",
    "Mean_OD_in", "Mean_ID_in", "MeanNetArea_in2",
    "MeanShear_ksi", "MeanShearLoad_lbf", "SpecimenCount"
]
display_cols = [c for c in display_cols if c in processed_df.columns]

display(processed_df[display_cols])


## Basic data quality checks

In [ ]:
print("Missing values by column:")
missing = processed_df.isna().sum()
display(missing[missing > 0])

print("\nRows by lot:")
display(processed_df["Lot_ID"].value_counts())

print("\nRows by test stage:")
display(processed_df["TestStage"].value_counts())
